# Exercise 3: Disambiguation in context

*Neural Morphology · version 0.1.0*

> The Turkish form *yüz* has three readings in the chapter's *yüz* table. Collect at least 60 sentences per reading containing the bare form *yüz* (from several documents), and split them by document into 40 training, 10 development and 10 test sentences per reading. Fine-tune a Turkish BERT classifier. What accuracy does it achieve on the test set? Which contextual cues (a following noun, a preceding adjective, sentence-final position) signal each reading?

**Setup.** `pip install "turkicnlp[hfst]==0.5.0" "transformers==5.3.0" sentencepiece protobuf scikit-learn stanza requests` (the compiled `hfst` package needs Python 3.11 or 3.12). Tokenisers are loaded from Hugging Face, or from a local copy in TurkicNLP's model folder (`~/.turkicnlp/models/huggingface/<org>--<name>`). Universal Dependencies files are read from the UD GitHub repositories (master branch), so internet access is needed. Cells whose model cannot be loaded print the reason instead of a result.

Annotated sentences are the cheapest source: the Turkish UD treebanks label every occurrence of *yüz* with its part of speech. The first cell collects all bare occurrences from seven treebanks and counts them by reading. The second trains a small logistic-regression classifier on the neighbouring words and holds out whole documents, so that sentences from the same document are not split between training and test. A fine-tuned BERT classifier is the next step once enough examples per reading exist.

In [ ]:
import os, re, collections, statistics
from pathlib import Path
import requests

UD = "https://raw.githubusercontent.com/UniversalDependencies/{repo}/master/{file}"
WORD = re.compile(r"[^\W_]+")                         # the book's word definition (Scripts and Encoding)

def ud_sentences(repo, file):
    """CoNLL-U sentences as lists of (form, lemma, upos, feats) for integer-ID words, with the sent_id."""
    text = requests.get(UD.format(repo=repo, file=file), timeout=120)
    text.raise_for_status()
    sents, words, sid = [], [], None
    for line in text.text.splitlines():
        if line.startswith("# sent_id"):
            sid = line.split("=", 1)[1].strip()
        elif not line.strip():
            if words:
                sents.append((sid, words))
            words = []
        elif not line.startswith("#"):
            c = line.split("\t")
            if c[0].isdigit():
                words.append((c[1], c[2], c[3], c[5]))
    if words:
        sents.append((sid, words))
    return sents

def load_tokenizer(name):
    """A Hugging Face tokenizer, from TurkicNLP's local model folder if present, otherwise from the Hub."""
    from transformers import AutoTokenizer
    local = Path("~/.turkicnlp/models/huggingface").expanduser() / name.replace("/", "--")
    return AutoTokenizer.from_pretrained(str(local) if (local / "tokenizer_config.json").exists() else name)

def pieces(tok, word):
    """Subword pieces of one word without special tokens and without the word-initial marker."""
    out = []
    for p in tok.tokenize(word):
        p = p.replace("▁", "").replace("Ġ", "")
        out.append(p[2:] if p.startswith("##") else p)
    return [p for p in out if p]

CORE_FEATS = ("Case", "Number", "Number[psor]", "Person", "Person[psor]", "Tense", "VerbForm", "Voice")
DEFAULTS = {("Case", "Nom"), ("Number", "Sing"), ("Person", "3")}

def core(upos, feats):
    """UPOS plus the core features, without the unmarked defaults Case=Nom, Number=Sing and Person=3,
    and with Polarity only if negative. Treebanks differ in whether they write the defaults out."""
    kv = dict(f.split("=", 1) for f in (feats or "").split("|") if "=" in f)
    keep = {k: v for k, v in kv.items() if k in CORE_FEATS and (k, v) not in DEFAULTS}
    if kv.get("Polarity") == "Neg":
        keep["Polarity"] = "Neg"
    return upos + "|" + "|".join(f"{k}={v}" for k, v in sorted(keep.items()))

In [ ]:
TREEBANKS = {"IMST": ["train", "dev", "test"], "BOUN": ["train", "dev", "test"], "Kenet": ["train", "dev", "test"],
             "Penn": ["train", "dev", "test"], "Tourism": ["train", "dev", "test"], "FrameNet": ["train", "dev", "test"],
             "GB": ["test"]}
examples = []                                          # (group, previous word, next word, gold UPOS)
for tb, splits in TREEBANKS.items():
    for split in splits:
        for sid, ws in ud_sentences(f"UD_Turkish-{tb}", f"tr_{tb.lower()}-ud-{split}.conllu"):
            for i, (form, lemma, upos, feats) in enumerate(ws):
                if form.lower() == "yüz":
                    prev_w = ws[i - 1][0].lower() if i > 0 else "<s>"
                    next_w = ws[i + 1][0].lower() if i + 1 < len(ws) else "</s>"
                    doc = re.split(r"[-_.:]", sid or "x")[0]
                    examples.append((f"{tb}/{doc}", prev_w, next_w, upos))
print("bare 'yüz' by gold UPOS:", collections.Counter(e[3] for e in examples))

In [ ]:
import random
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

data = [e for e in examples if e[3] in ("NOUN", "NUM")]
groups = sorted({e[0] for e in data})
random.Random(13).shuffle(groups)
test_groups = set(groups[: len(groups) // 5])            # about a fifth of the documents for testing
train = [e for e in data if e[0] not in test_groups]
test = [e for e in data if e[0] in test_groups]

def features(prev_w, next_w):
    return {"prev=" + prev_w: 1, "next=" + next_w: 1, "next_suffix=" + next_w[-2:]: 1,
            "next_is_digit": int(next_w.isdigit()), "prev_is_digit": int(prev_w.isdigit()),
            "next_is_punct": int(not next_w.isalnum())}

vec = DictVectorizer()
X = vec.fit_transform([features(p, n) for _, p, n, _ in train])
clf = LogisticRegression(max_iter=2000, C=1.0).fit(X, [y for *_, y in train])
pred = clf.predict(vec.transform([features(p, n) for _, p, n, _ in test]))
gold = [y for *_, y in test]
majority = collections.Counter(y for *_, y in train).most_common(1)[0][0]
print(f"train {len(train)} ({collections.Counter(y for *_, y in train)}), test {len(test)} ({collections.Counter(gold)}) "
      f"from {len(test_groups)} held-out documents")
print(f"accuracy {sum(p == g for p, g in zip(pred, gold)) / len(gold):.3f}; "
      f"majority class ({majority}) {sum(g == majority for g in gold) / len(gold):.3f}")
weights = sorted(zip(clf.coef_[0], vec.get_feature_names_out()))
print("strongest cues for", clf.classes_[0], [f for _, f in weights[:6]])
print("strongest cues for", clf.classes_[1], [f for _, f in weights[-6:]])

## Worked solution

Output of `experiments/ch07_exercises/run_ch07_exercises.py` on 19 October 2026:

```
bare 'yüz' by gold UPOS: Counter({'NUM': 114, 'NOUN': 86, 'AUX': 3})
```

```
train 162 (Counter({'NUM': 94, 'NOUN': 68})), test 38 (Counter({'NUM': 20, 'NOUN': 18})) from 28 held-out documents
accuracy 0.816; majority class (NUM) 0.526
strongest cues for NOUN ['next=yüze', 'next_suffix=ze', 'prev=bir', 'prev=güler', 'prev=ters', 'prev=,']
strongest cues for NUM ['next=seksen', 'prev=yüzde', 'prev=dokuz', 'prev=<s>', 'next=bin', 'next_suffix=in']
```

The Turkish UD treebanks contain 203 bare occurrences of *yüz*: 114 numerals, 86 nouns, 3 tagged AUX, and **no verb**. The imperative 'swim!' does not occur at all, so the exercise's design (60 sentences per reading) cannot be met from treebanks for the verb reading; it needs another source and hand labelling (Your turn, 1). A frequent reading in one source can be absent in another, which is why the collection step comes before any model.

For the two readings that exist, a logistic regression on the neighbouring words, trained on 162 examples and tested on 38 from 28 held-out documents, reaches 81.6% accuracy, against 52.6% for always predicting the more frequent class (numeral). The strongest cues are what one expects: number words around it (*dokuz* 'nine' before, *seksen* 'eighty' or *bin* 'thousand' after) and *yüzde* 'percent' for the numeral; the article *bir* before it and case forms of the same noun nearby (*yüze*) for the noun 'face'. With 38 test examples the accuracy has a wide uncertainty (roughly ±12 points as a 95% interval), and a fine-tuned BERT classifier remains to be compared on the same split.

## Your turn — extension

1. Collect verb examples (*Havuzda yüz!*, *Her sabah yüz.*) from another source, for example by searching a web corpus for *yüz* before *!* or after *her gün*, and label them by hand.
2. With 60 examples per reading, replace the logistic regression by a fine-tuned BERTurk classifier (`AutoModelForSequenceClassification`, with the position of *yüz* marked) and compare on the same document split.